# Week 3 — Synthetic Data Generator (Anime × F1 × Cloud Security)

Generate synthetic datasets for **Anime**, **F1**, or **Cloud security** using Hugging Face (Llama 3.1).

Uses `HF_TOKEN`. In Colab: add `HF_TOKEN` in Secrets, then run all cells. Install: `!pip install -q python-dotenv huggingface_hub gradio`

In [ ]:
# Run this cell in Google Colab if needed
# !pip install -q python-dotenv huggingface_hub gradio

In [ ]:
import os
import re
from dotenv import load_dotenv
from huggingface_hub import login, InferenceClient
import gradio as gr

In [ ]:
load_dotenv(override=True)
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    print("HF_TOKEN found.")
    login(token=hf_token, add_to_git_credential=True)
else:
    print("Set HF_TOKEN in .env or Colab secrets.")

MODELS = {
    "Llama 3.1": "meta-llama/Meta-Llama-3.1-8B-Instruct",
}

In [ ]:
SYSTEM_PROMPT = """You are a synthetic dataset generator. Output only structured data.

Rules:
- Output ONLY CSV or JSON. No explanations, no markdown code fences.
- CSV: first line = header, then one row per record.
- JSON: one array of objects, same keys in each object.
- All data must be fake (no real names, real anime titles, real drivers, real account IDs).
- Generate exactly the number of records requested."""

In [ ]:
# Presets: description and columns for each dataset type
DATASET_PRESETS = {
    "Anime catalog": {
        "description": "Anime catalog with fake titles and studios.",
        "columns": "title, genre, studio, season_year, episodes, rating (1-10), status (watching/completed/dropped)",
    },
    "F1 race results": {
        "description": "F1 race results with fake driver and team names, plausible circuits and countries.",
        "columns": "grand_prix, circuit, country, date, driver, team, position, points",
    },
    "Cloud security events": {
        "description": "Cloud security events (S3, IAM, EC2) with fake account IDs.",
        "columns": "event_id, timestamp, resource_type, event_type, severity, account_id, region, remediation",
    },
}

In [ ]:
def generate_data(model_name, dataset_type, row_count, output_format):
    """Generate synthetic data using Hugging Face InferenceClient."""
    if not dataset_type or dataset_type not in DATASET_PRESETS:
        return "Please choose a dataset type."
    preset = DATASET_PRESETS[dataset_type]
    description = preset["description"]
    columns = preset["columns"]
    row_count = max(1, min(int(row_count), 40))
    fmt = output_format.strip().upper() or "CSV"

    user_prompt = (
        f"Generate a dataset: {description} "
        f"Columns: {columns}. "
        f"Exactly {row_count} rows. Output format: {fmt} only, no other text."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    try:
        model_id = MODELS[model_name]
        client = InferenceClient(model=model_id, token=hf_token)
        response = client.chat_completion(
            messages=messages,
            max_tokens=2000,
            temperature=0.3,
        )
        raw = (response.choices[0].message.content or "").strip()
        if raw.startswith("```"):
            raw = re.sub(r"^```\w*\n", "", raw)
            raw = re.sub(r"\n```\s*$", "", raw)
        return raw
    except Exception as e:
        return f"Error: {e}"

In [ ]:
with gr.Blocks(title="Synthetic Data Generator") as demo:
    gr.Markdown("## Synthetic Data Generator — Anime, F1, Cloud Security")
    model_dropdown = gr.Dropdown(
        choices=list(MODELS.keys()),
        value="Llama 3.1",
        label="Model",
    )
    dataset_dropdown = gr.Dropdown(
        choices=list(DATASET_PRESETS.keys()),
        value="Anime catalog",
        label="Dataset type",
    )
    row_count = gr.Slider(1, 40, value=5, step=1, label="Number of rows")
    output_format = gr.Radio(["CSV", "JSON"], value="CSV", label="Format")
    btn = gr.Button("Generate", variant="primary")
    out = gr.Textbox(label="Generated data", lines=14)

    btn.click(
        fn=generate_data,
        inputs=[model_dropdown, dataset_dropdown, row_count, output_format],
        outputs=out,
    )

demo.launch()